In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import seaborn as sns

In [2]:
world = gpd.read_file('./data/shp/territory_vis.shp')


In [9]:
def add_colorbar(fig,ax,vmax,label,vmin=0,cmap='autumn_r',loc=[0.175,0.39,0.02,0.2],extend='max'):
        
    cax = ax.inset_axes(loc,transform=ax.transAxes)
    
    im = plt.cm.ScalarMappable(cmap=cmap,
                               norm=plt.Normalize(vmin=vmin,
                                                  vmax=vmax))
    
    if extend == 'none':
        cbar = fig.colorbar(im,cax=cax)
    else:
        cbar = fig.colorbar(im,cax=cax,extend=extend)
    
    cbar.outline.set_edgecolor('none')
    
    cbar.ax.tick_params(labelsize=15) 
    
    cbar.minorticks_on()
    
    cax.yaxis.tick_left()
    
    cbar.set_label(label,font={'size':25})
    


def draw_attr_map(fig,ax,geo,col,cmap,vmax,cbar_label,cbar_loc,draw_edge=False,extend='max'):
    world.plot(ax=ax,
               facecolor='whitesmoke',
               edgecolor='silver',
               lw=0.25)
    
    if draw_edge:
        geo.plot(ax=ax,
                column=col,
                cmap=cmap,
                vmax=vmax,
                edgecolor='silver',
                linewidth=0.25)
    else:
        geo.plot(ax=ax,
                column=col,
                cmap=cmap,
                vmax=vmax)
        
    
    add_colorbar(fig=fig,
                 ax=ax,
                 label=cbar_label,
                 vmax=vmax,
                 cmap=cmap,
                 loc=cbar_loc,
                 extend=extend)
    
    

In [13]:
grid_attr = pd.read_csv('./data/grid.csv')

In [14]:
pv_res = gpd.read_file('./data/vre_res/pv_res.gpkg')
we_res = gpd.read_file('./data/vre_res/we_res.gpkg')

pv_res = pv_res.loc[pv_res['tot']>0]
we_res = we_res.loc[we_res['tot']>0]

In [15]:
upv_res = pd.read_csv('./data/vre_res/upv_attr_era5_cell_2019.csv')
dpv_res = pd.read_csv('./data/vre_res/dpv_attr_era5_cell_2019.csv')
onshore_res = pd.read_csv('./data/vre_res/onshore_attr_era5_cell_2019.csv')
offshore_res = pd.read_csv('./data/vre_res/offshore_attr_era5_cell_2019.csv')


In [16]:
onshore_res['to_lc_dis'] = onshore_res['to_sub_dis']+onshore_res['sub_urban_dis']
offshore_res['to_lc_dis'] = offshore_res['to_sub_dis']+offshore_res['sub_urban_dis']
upv_res['to_lc_dis'] = upv_res['to_sub_dis']+upv_res['sub_urban_dis']
dpv_res['to_lc_dis'] = 0.0

pv_res_df = pd.concat([upv_res,dpv_res])
we_res_df = pd.concat([onshore_res,offshore_res])


In [17]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 18

fig,axes = plt.subplots(2,1,figsize=(24,20),dpi=800)


plt.subplots_adjust(
    hspace=0.0   # 子图间垂直间距（高度比例）
)

ax = axes[0]

draw_attr_map(fig=fig,
              ax=ax,
              geo=we_res,
              col='tot',
              cmap='viridis_r',
              cbar_label='Capacity (GW)',
              vmax=2.5,
              cbar_loc=[0.05,0.275,0.02,0.4])

ax.text(0.015,
        0.95,
        'a',
        ha='center',
        va='center',
        transform=ax.transAxes,
        size = 30,
        weight='bold')

ax.set_xticks([],[])
ax.set_yticks([],[])

ax.axis('off')

ax = axes[1]

draw_attr_map(fig=fig,
              ax=ax,
              geo=pv_res,
              col='tot',
              cmap='autumn_r',
              cbar_label='Capacity (GW)',
              vmax=5.5,
              cbar_loc=[0.05,0.275,0.02,0.4])

ax.text(0.015,
        0.95,
        'b',
        ha='center',
        va='center',
        transform=ax.transAxes,
        size = 30,
        weight='bold')

ax.axis('off')

ax.set_xticks([],[])
ax.set_yticks([],[])

plt.savefig('./fig/vre_res.jpg',bbox_inches='tight')
plt.savefig('./fig/vre_res.pdf',bbox_inches='tight')

plt.close()


In [18]:
europe = list(grid_attr.loc[grid_attr['irena_code']=='Europe']['region'])

row = list(grid_attr.loc[(grid_attr['irena_code']!='Europe')&
                         (grid_attr['region']!='China')&
                         (grid_attr['region']!='USA')&
                         (grid_attr['region']!='India')]['region'])

In [19]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 20

fig,axes = plt.subplots(1,2,figsize=(24,6),dpi=600)

plt.subplots_adjust(
    wspace=0.075  
)

ax = axes[0]

sns.ecdfplot(data=we_res_df,
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='Global',
             color='black',
             lw=2.5)

sns.ecdfplot(data=we_res_df.loc[we_res_df['region']=='China'],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='China',
             color='red',
             lw=1.5)

sns.ecdfplot(data=we_res_df.loc[we_res_df['region']=='USA'],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='USA',
             color='#20918d',
             lw=1.5)

sns.ecdfplot(data=we_res_df.loc[we_res_df['region'].isin(europe)],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='Europe',
             color='#ff68b5',
             lw=1.5)

sns.ecdfplot(data=we_res_df.loc[we_res_df['region']=='India'],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='India',
             color='#a65648',
             lw=1.5)


sns.ecdfplot(data=we_res_df.loc[we_res_df['region'].isin(row)],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='Other',
             color='#8a2ae3',
             lw=1.5)


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel('Distance (km)')
ax.set_ylabel('Proportion (0~1)')
ax.grid(visible=True,which='both',ls='-.',lw=0.5)
ax.set_title('Wind power')

ax.text(-0.1,
        1.025,
        'c',
        ha='center',
        va='center',
        transform=ax.transAxes,
        size = 30,
        weight='bold')

ax.minorticks_on()

ax.tick_params(axis='y', 
               which='major', 
               length=8, 
               color='black')

ax.tick_params(axis='x', 
               which='minor', 
               length=2.5, 
               color='blue')

ax = axes[1]

sns.ecdfplot(data=pv_res_df,
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             color='black',
             label='Global',
             lw=2.5)

sns.ecdfplot(data=pv_res_df.loc[pv_res_df['region']=='China'],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='China',
             color='red',
             lw=1.5)

sns.ecdfplot(data=pv_res_df.loc[pv_res_df['region']=='USA'],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='USA',
             color='#20918d',
             lw=1.5)

sns.ecdfplot(data=pv_res_df.loc[pv_res_df['region'].isin(europe)],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='Europe',
             color='#ff68b5',
             lw=1.5)

sns.ecdfplot(data=pv_res_df.loc[pv_res_df['region']=='India'],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='India',
             color='#a65648',
             lw=1.5)


sns.ecdfplot(data=pv_res_df.loc[pv_res_df['region'].isin(row)],
             x='to_lc_dis',
             weights='tot',
             ax=ax,
             log_scale=True,
             label='Other',
             color='#8a2ae3',
             lw=1.5)

ax.legend()

ax.set_xlabel('Distance (km)')
ax.set_ylabel('')

ax.set_title('Solar PV')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(visible=True,which='both',ls='-.',lw=0.5)

ax.minorticks_on()

ax.tick_params(axis='y', 
               which='major', 
               length=8, 
               color='black')

ax.tick_params(axis='x', 
               which='minor', 
               length=2.5, 
               color='blue')

plt.savefig('./fig/vre_ecdf_dis.jpg',bbox_inches='tight')
plt.savefig('./fig/vre_ecdf_dis.pdf',bbox_inches='tight')

plt.close()
